In [1]:
import pathlib
import sys
import duckdb
import pandas as pd
import ipywidgets as w
from IPython.display import display, HTML
from dotenv import load_dotenv


_ROOT = pathlib.Path().resolve()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(_ROOT / "src"))
from irp.features.metrics import compute_metrics

load_dotenv()
con = duckdb.connect(str(_ROOT / "data/irp.duckdb"))
con.execute("PRAGMA disable_progress_bar")

fundamentals = {
    t: con.execute(f"SELECT * FROM {t}").df() for t in ("income", "balance", "cashflow")
}
# Restrict prices to tickers that have fundamentals (screener needs both)
prices = con.execute(
    'SELECT ticker, date, open, high, low, close, volume FROM prices '
    'WHERE ticker IN (SELECT DISTINCT "Ticker" FROM income)'
).df()

ticker_info = con.execute("""
    SELECT c.Ticker AS ticker, i.Sector AS sector, i.Industry AS industry
    FROM companies c
    LEFT JOIN industries i ON c.IndustryId = i.IndustryId
""").df()

metrics_cache = {}


def get_metrics(period: str) -> pd.DataFrame:
    if period not in metrics_cache:
        metrics_cache[period] = compute_metrics(
            fundamentals, prices, period, latest=True
        )
    return metrics_cache[period]


_warm = get_metrics("annual")
print(f"Loaded {_warm.shape[0]} tickers, {_warm.shape[1]} metric columns")

import irp.ticker_sets as ts
ts.init(con)

Loaded 4420 tickers, 48 metric columns


In [2]:
# (column name, display format) — drives both UI dropdown and table rendering
METRIC_DEFS = [
    # size
    ("Market Cap",            "$B"),
    ("Enterprise Value",      "$B"),
    # valuation
    ("P/E",                   "ratio"),
    ("PEG",                   "ratio"),
    ("Earnings Yield",        "pct"),
    ("P/B",                   "ratio"),
    ("P/S",                   "ratio"),
    ("P/FCF",                 "ratio"),
    ("EV/EBITDA",             "ratio"),
    ("EV/Sales",              "ratio"),
    # profitability
    ("Gross Margin",          "pct"),
    ("Operating Margin",      "pct"),
    ("Net Margin",            "pct"),
    ("ROE",                   "pct"),
    ("ROA",                   "pct"),
    ("ROIC",                  "pct"),
    # growth
    ("Revenue Growth YoY",    "pct"),
    ("Revenue 3Y CAGR",       "pct"),
    ("EPS Growth YoY",        "pct"),
    ("EPS 3Y CAGR",           "pct"),
    ("FCF Growth YoY",        "pct"),
    ("FCF 3Y CAGR",           "pct"),
    ("OpInc Growth YoY",      "pct"),
    ("OpInc 3Y CAGR",         "pct"),
    # health
    ("Debt/Equity",           "ratio"),
    ("Net Debt/EBITDA",       "ratio"),
    ("Current Ratio",         "ratio"),
    ("Quick Ratio",           "ratio"),
    ("Interest Coverage",     "ratio"),
    # cash flow
    ("FCF Yield",             "pct"),
    ("FCF Margin",            "pct"),
    ("Cash Conversion",       "ratio"),
    # dividend
    ("Dividend Yield",        "pct"),
    ("Payout Ratio",          "pct"),
    # momentum
    ("Return 1M",             "pct"),
    ("Return 3M",             "pct"),
    ("Return 6M",             "pct"),
    ("Return 1Y",             "pct"),
    ("Return YTD",            "pct"),
    ("Dist from 52w High",    "pct"),
    ("Dist from 52w Low",     "pct"),
    ("Volatility (annualized)", "pct"),
    # scores
    ("Piotroski F",           "int"),
    ("Altman Z",              "ratio"),
]
METRIC_NAMES = [m[0] for m in METRIC_DEFS]
METRIC_FMT   = dict(METRIC_DEFS)

DEFAULT_COLS = ["Market Cap", "P/E", "PEG", "P/B", "ROE", "Net Margin",
                "Revenue 3Y CAGR", "Debt/Equity", "FCF Yield",
                "Dividend Yield", "Return 1Y", "Piotroski F"]

def _fmt_cell(v, kind):
    if pd.isna(v): return ""
    if kind == "pct":   return f"{v:,.1%}"
    if kind == "ratio": return f"{v:,.2f}"
    if kind == "int":   return f"{int(v)}"
    if kind == "$B":    return f"${v:,.0f}B"
    return str(v)

_STYLE = """<style>
table.scr { border-collapse: collapse; font-size: 12px; color: #e0e0e0; background: #141414; }
table.scr th { text-align: right; padding: 4px 8px; border-bottom: 1px solid #555;
               position: sticky; top: 0; background: #1e1e1e; color: #e0e0e0; }
table.scr td { text-align: right; padding: 3px 8px; white-space: nowrap;
               background: #ffffff; color: #000000; }
table.scr tr:nth-child(even) td { background: #e6e6e6; }
table.scr td:first-child, table.scr th:first-child { text-align: left; font-weight: bold; }
table.scr tr:hover td { background: #2a2a2a; color: #e0e0e0; }
</style>"""

In [3]:
import re
import webbrowser
from ipyevents import Event

# --- top controls ---
period_dd = w.Dropdown(
    options=[("Annual", "annual"), ("TTM", "ttm")],
    value="annual", description="Period:",
    style={"description_width": "initial"},
)
cols_sel = w.SelectMultiple(
    options=METRIC_NAMES, value=tuple(DEFAULT_COLS),
    description="Show columns:", rows=12,
    layout=w.Layout(width="320px"),
    style={"description_width": "initial"},
)
sort_dd = w.Dropdown(
    options=METRIC_NAMES, value="P/E",
    description="Sort by:", style={"description_width": "initial"},
)
sort_asc = w.Checkbox(value=True, description="Ascending")
limit_in = w.IntText(value=50000, description="Limit:", layout=w.Layout(width="140px"),
                    style={"description_width": "initial"})

# --- sector / industry ---
_all_sectors    = ["(All)"] + sorted(ticker_info["sector"].dropna().unique().tolist())
_all_industries = ["(All)"] + sorted(ticker_info["industry"].dropna().unique().tolist())
sector_sel = w.SelectMultiple(
    options=_all_sectors, value=["(All)"],
    description="Sector:", rows=8,
    layout=w.Layout(width="260px"),
    style={"description_width": "initial"},
)
industry_sel = w.SelectMultiple(
    options=_all_industries, value=["(All)"],
    description="Industry:", rows=8,
    layout=w.Layout(width="300px"),
    style={"description_width": "initial"},
)

# --- watchlist ---
_watchlist: list[str] = []
watchlist_sel = w.SelectMultiple(
    options=[], description="Watchlist:", rows=8,
    layout=w.Layout(width="260px"),
    style={"description_width": "initial"},
)
rm_wl_btn = w.Button(description="✕ Remove", button_style="warning",
                     layout=w.Layout(width="90px"))
yf_btn = w.Button(description="Open Yahoo Finance", button_style="info",
                  layout=w.Layout(width="150px"))

def _rm_from_watchlist(_=None):
    for t in list(watchlist_sel.value):
        _watchlist.remove(t)
    watchlist_sel.options = tuple(_watchlist)

def _open_yahoo_finance(_=None):
    targets = list(watchlist_sel.value) or _watchlist
    for ticker in targets:
        webbrowser.open(f"https://finance.yahoo.com/quote/{ticker}/")

rm_wl_btn.on_click(_rm_from_watchlist)
yf_btn.on_click(_open_yahoo_finance)

def _add_to_watchlist(ticker):
    if ticker and ticker not in _watchlist:
        _watchlist.append(ticker)
        watchlist_sel.options = tuple(_watchlist)

# --- table widget + ipyevents click listener ---
table_widget = w.HTML(value="<em style='color:#888;font-size:12px'>Press Apply to see results.</em>")
_click_listener = Event(source=table_widget, watched_events=["click"])

def _on_table_click(event):
    target = event.get("target", {})
    if "scr-ticker" in target.get("className", ""):
        elem_id = target.get("id", "")
        if elem_id.startswith("ticker-"):
            _add_to_watchlist(elem_id[len("ticker-"):])

_click_listener.on_dom_event(_on_table_click)

def _make_tickers_clickable(html: str) -> str:
    if "<tbody>" not in html:
        return html
    pre, rest = html.split("<tbody>", 1)
    body, post = rest.split("</tbody>", 1)

    def _process_row(m):
        row = m.group(0)
        td_m = re.search(r'<td[^>]*>([^<]+)</td>', row)
        if not td_m:
            return row
        ticker = td_m.group(1).strip()
        return re.sub(
            r'<td[^>]*>[^<]+</td>',
            f'<td id="ticker-{ticker}" class="scr-ticker"'
            f' style="cursor:pointer;font-weight:bold;color:#58a6ff;text-decoration:underline">'
            f'{ticker}</td>',
            row,
            count=1,
        )

    body = re.sub(r'<tr[^>]*>.*?</tr>', _process_row, body, flags=re.DOTALL)
    return pre + "<tbody>" + body + "</tbody>" + post

# --- dynamic metric filter list ---
filter_rows: list[dict] = []
filters_box = w.VBox([])

def _add_filter(_=None, metric: str | None = None, vmin: float | None = None, vmax: float | None = None):
    metric_dd = w.Dropdown(options=METRIC_NAMES, value=metric or METRIC_NAMES[0],
                           layout=w.Layout(width="240px"))
    min_in = w.FloatText(value=vmin, description="min", layout=w.Layout(width="160px"),
                         style={"description_width": "initial"})
    max_in = w.FloatText(value=vmax, description="max", layout=w.Layout(width="160px"),
                         style={"description_width": "initial"})
    rm_btn = w.Button(description="x", layout=w.Layout(width="32px"), button_style="warning")
    row = w.HBox([metric_dd, min_in, max_in, rm_btn])
    entry = {"box": row, "metric": metric_dd, "min": min_in, "max": max_in}
    filter_rows.append(entry)
    rm_btn.on_click(lambda _b: _remove_filter(entry))
    filters_box.children = tuple(e["box"] for e in filter_rows)

def _remove_filter(entry):
    filter_rows.remove(entry)
    filters_box.children = tuple(e["box"] for e in filter_rows)

add_btn   = w.Button(description="+ Add filter", button_style="info")
add_btn.on_click(_add_filter)
apply_btn = w.Button(description="Apply", button_style="primary")

def _apply(_=None):
    try:
        df = get_metrics(period_dd.value).copy()
        n0 = len(df)

        sel_sectors    = [s for s in sector_sel.value   if s != "(All)"]
        sel_industries = [i for i in industry_sel.value if i != "(All)"]
        if sel_sectors or sel_industries:
            info = ticker_info.copy()
            if sel_sectors:    info = info[info["sector"].isin(sel_sectors)]
            if sel_industries: info = info[info["industry"].isin(sel_industries)]
            df = df[df["ticker"].isin(info["ticker"])]

        for e in filter_rows:
            col = e["metric"].value
            lo, hi = e["min"].value, e["max"].value
            if not pd.isna(lo): df = df[df[col] >= lo]
            if not pd.isna(hi): df = df[df[col] <= hi]

        df = df.sort_values(sort_dd.value, ascending=sort_asc.value, na_position="last")
        df = df.head(limit_in.value)

        cols = ["ticker", "period"] + list(cols_sel.value)
        view = df[cols].copy()
        for c in cols_sel.value:
            view[c] = view[c].apply(lambda v, k=METRIC_FMT[c]: _fmt_cell(v, k))

        raw_table = view.to_html(index=False, classes="scr", escape=False)
        clickable  = _make_tickers_clickable(raw_table)
        count = f'<p style="color:#aaa;font-size:12px;margin:4px 0">{len(df)} of {n0} tickers match — click ticker to add to watchlist</p>'
        table_widget.value = (
            _STYLE + count
            + '<div style="max-height:500px;overflow-y:auto;border:1px solid #333">'
            + clickable + "</div>"
        )
    except Exception as e:
        table_widget.value = f'<pre style="color:#f88">Error: {e}</pre>'

apply_btn.on_click(_apply)

_add_filter(metric="P/E",  vmin=0,    vmax=20.0)
_add_filter(metric="ROE",  vmin=0.15, vmax=9999999)


# --- ticker sets ---
set_name_in = w.Text(
    placeholder="Set name (e.g. fast_growing)",
    layout=w.Layout(width="220px"),
)
set_mode_radio = w.RadioButtons(
    options=["Replace", "Add to"],
    value="Replace",
    layout=w.Layout(width="100px"),
)
save_set_btn = w.Button(
    description="Save as Set", button_style="success",
    layout=w.Layout(width="110px"),
)
set_status_lbl = w.Label("")

sets_dd = w.Dropdown(
    options=[f"{name} ({n})" for name, n in ts.list_sets(con)] or ["(none)"],
    description="Existing sets:",
    layout=w.Layout(width="300px"),
    style={"description_width": "initial"},
)
load_set_btn = w.Button(
    description="Load to Watchlist", button_style="info",
    layout=w.Layout(width="140px"),
)
delete_set_btn = w.Button(
    description="Delete", button_style="danger",
    layout=w.Layout(width="80px"),
)

def _refresh_sets():
    opts = [f"{name} ({n})" for name, n in ts.list_sets(con)]
    sets_dd.options = opts or ["(none)"]

def _save_set(_=None):
    name = set_name_in.value.strip()
    if not name:
        set_status_lbl.value = "Enter a set name first."
        return
    if not _watchlist:
        set_status_lbl.value = "Add tickers to watchlist first."
        return
    ts.save_set(name, _watchlist, con, replace=(set_mode_radio.value == "Replace"))
    _refresh_sets()
    set_status_lbl.value = f"Saved {len(_watchlist)} tickers to \'{name}\'."

def _load_set(_=None):
    if not sets_dd.value or sets_dd.value == "(none)":
        return
    set_name = sets_dd.value.split(" (")[0]
    tickers = ts.get_set(set_name, con)
    for t in tickers:
        _add_to_watchlist(t)
    set_status_lbl.value = f"Loaded {len(tickers)} tickers from \'{set_name}\' to watchlist."

def _delete_set(_=None):
    if not sets_dd.value or sets_dd.value == "(none)":
        return
    set_name = sets_dd.value.split(" (")[0]
    ts.delete_set(set_name, con)
    _refresh_sets()
    set_status_lbl.value = f"Deleted set \'{set_name}\'."

save_set_btn.on_click(_save_set)
load_set_btn.on_click(_load_set)
delete_set_btn.on_click(_delete_set)

display(
    w.HBox([w.VBox([period_dd, sort_dd, sort_asc, limit_in]), cols_sel]),
    w.HBox([
        w.VBox([w.Label("Sector (empty = all):"), sector_sel]),
        w.VBox([w.Label("Industry (empty = all):"), industry_sel]),
        w.VBox([w.Label("Watchlist:"), watchlist_sel, w.HBox([rm_wl_btn, yf_btn])]),
    ]),
    w.VBox([
        w.Label("Save filtered results as Ticker Set:"),
        w.HBox([set_name_in, set_mode_radio, save_set_btn]),
        w.HBox([sets_dd, load_set_btn, delete_set_btn]),
        set_status_lbl,
    ]),
    w.Label("Metric filters (AND-joined):"),
    filters_box,
    w.HBox([add_btn, apply_btn]),
    table_widget,
)

Label(value='Metric filters (AND-joined):')

HTML(value="<em style='color:#888;font-size:12px'>Press Apply to see results.</em>")

In [5]:
_watchlist

['GDDY',
 'GEN',
 'ARW',
 'ADBE',
 'META',
 'NOK',
 'QCOM',
 'STM',
 'VOD',
 'NSIT',
 'TDC']